# Entendiendo la Bellman Equation en Q-Learning

Este notebook desglosa el núcleo de Q-Learning: la **Bellman Equation**. Iremos paso a paso para ver cómo se actualiza el "cerebro" del Agent —la Q-table—. Usaremos un ejemplo simple inspirado en el Environment `FrozenLake`.

## ¿Qué es la Bellman Equation?

La Bellman Equation es la regla de actualización usada en Q-learning. Nos dice cómo mejorar nuestra estimación del valor de un par State-Action basándonos en la nueva información que recopilamos del Environment.

Aquí está la fórmula:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

Parece compleja, pero desglosémosla en términos simples:

**`Nuevo Q-value = Antiguo Q-value + Tasa de Aprendizaje * (Error de Diferencia Temporal)`**

Donde:
*   **`Q(s, a)`**: El Q-value actual para estar en el **State `s`** y tomar la **Action `a`**. Esta es nuestra estimación antigua.
*   **`α` (alpha)**: La **Tasa de Aprendizaje** (p.ej., 0.1). Controla cuánto actualizamos nuestro Q-value. Un `α` pequeño significa que aprendemos lentamente, mientras que un `α` grande significa que aprendemos rápido.
*   **`r`**: El **Reward** inmediato que obtenemos después de tomar la Action `a` desde el State `s`.
*   **`γ` (gamma)**: El **Factor de Descuento** (p.ej., 0.95). Determina cuánto valoramos los Rewards futuros. Un `γ` cercano a 1 significa que nos importan mucho los Rewards futuros, mientras que un `γ` cercano a 0 significa que solo nos importa el Reward inmediato.
*   **`max Q(s', a')`**: El **mejor Q-value posible** que podemos obtener desde nuestro **siguiente State `s'`**. Esto es el Agent mirando un paso adelante y eligiendo la Action más valiosa desde su nueva posición.
*   **`r + γ * max Q(s', a')`**: Este es el **valor aprendido** o la "nueva y mejor estimación" del Q-value. Es el Reward inmediato más la estimación descontada del Reward futuro.
*   **`[...] - Q(s, a)`**: Esta parte completa entre corchetes es el **Error de Diferencia Temporal (TD Error)**. Es la diferencia entre nuestra nueva estimación y la antigua. Representa la "sorpresa" o qué tan equivocada estaba nuestra estimación inicial.

## Un Ejemplo Paso a Paso

Simulemos una pequeña parte del mundo `FrozenLake`. Imagina este simple Environment de 3 estados:

`Inicio (S0)` <-> `Congelado (S1)` <-> `Goal (S2)`


In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt

env = gym.make('FrozenLake-v1', desc=["SFG"], is_slippery=False, render_mode="rgb_array")
env.reset()

def show_env(env):
    '''Renderizar y mostrar el Environment en el notebook.'''
    img = env.render()
    plt.imshow(img)
    plt.axis('off')
    plt.show()

show_env(env)


*   **States**: S0, S1, S2
*   **Actions**: 0 (Izquierda), 1 (Derecha)
*   **Rewards**:
    *   +1 al alcanzar el Goal (S2)
    *   0 para cualquier otro movimiento.

Establezcamos nuestros hiperparámetros:
*   Tasa de Aprendizaje (`α`) = 0.1
*   Factor de Descuento (`γ`) = 0.9

### Q-Table Inicial

Nuestro Agent comienza sin ningún conocimiento. La Q-table es una matriz `[states x actions]` llena de ceros.

$$
\begin{array}{c c}
    & \color{gray} \begin{matrix}
    \text{Izquierda (0)} & \text{Derecha (1)} \end{matrix}
    \\
    \color{gray}
    \begin{matrix} \text{S0} \\ \text{S1} \\ \text{S2} \end{matrix}
    &
    \begin{bmatrix}
        0 & \quad 0 \\
        0 & \quad 0 \\
        0 & \quad 0
    \end{bmatrix}
\end{array}
$$

In [ ]:
import numpy as np

q_table = np.zeros((3, 2)) # 3 states, 2 actions

print("Q-Table Inicial:")
print(q_table)

### Paso 1: El Agent se mueve de S0 a S1

1.  **State actual (`s`)**: S0
2.  **Action (`a`)**: 1 (Moverse a la Derecha)
3.  **Observación**:
    *   Nuevo State (`s'`) = S1
    *   Reward (`r`) = 0

Ahora, apliquemos la Bellman Equation para actualizar `Q(S0, Derecha)`.

**Fórmula**: `Q(s, a) = Q(s, a) + α * [r + γ * max Q(s', a') - Q(s, a)]`

**Cálculo**:
*   `Q(S0, Derecha)` = `q_table[0, 1]` = **0**
*   `α` = **0.1**
*   `r` = **0**
*   `γ` = **0.9**
*   `max Q(S1, a')`: El mejor Q-value desde el State S1. Miramos la fila de S1 en nuestra Q-table (`q_table[1]`). Los valores son `[0, 0]`. El valor máximo es **0**.

Sustituyamos:
`Nuevo Q(S0, Derecha) = 0 + 0.1 * [0 + 0.9 * 0 - 0]`
`Nuevo Q(S0, Derecha) = 0`

No muy emocionante. El Q-value no cambió porque no se encontró ningún Reward. Esto es común al principio del training.

### Paso 2: El Agent se mueve de S1 a S2 (¡El Goal!)

1.  **State actual (`s`)**: S1
2.  **Action (`a`)**: 1 (Moverse a la Derecha)
3.  **Observación**:
    *   Nuevo State (`s'`) = S2 (Goal)
    *   Reward (`r`) = **+1**

Actualicemos `Q(S1, Derecha)`.

**Cálculo**:
*   `Q(S1, Derecha)` = `q_table[1, 1]` = **0**
*   `α` = **0.1**
*   `r` = **1**
*   `γ` = **0.9**
*   `max Q(S2, a')`: El mejor Q-value desde el State S2. La fila de S2 en nuestra Q-table (`q_table[2]`) es `[0, 0]`. El valor máximo es **0**. (S2 es un State terminal, por lo que los Rewards futuros desde aquí son 0).

Sustituyamos:
`Nuevo Q(S1, Derecha) = 0 + 0.1 * [1 + 0.9 * 0 - 0]`
`Nuevo Q(S1, Derecha) = 0.1 * [1]`
`Nuevo Q(S1, Derecha) = 0.1`

**¡Esta es nuestra primera actualización!** El Agent aprendió que moverse a la derecha desde S1 es ligeramente bueno.

In [ ]:
q_table[1, 1] = 0.1
print("Q-Table después del Paso 2:")
print(q_table)

### Paso 3: Nuevo Episodio - El Agent se mueve de S0 a S1 nuevamente

Ahora ocurre la magia. La información del Reward comienza a fluir hacia atrás.

1.  **State actual (`s`)**: S0
2.  **Action (`a`)**: 1 (Moverse a la Derecha)
3.  **Observación**:
    *   Nuevo State (`s'`) = S1
    *   Reward (`r`) = 0

Actualicemos `Q(S0, Derecha)` nuevamente.

**Cálculo**:
*   `Q(S0, Derecha)` = `q_table[0, 1]` = **0** (de antes)
*   `α` = **0.1**
*   `r` = **0**
*   `γ` = **0.9**
*   `max Q(S1, a')`: **¡Esta es la diferencia clave!** Miramos la fila de S1 en nuestra Q-table actualizada (`q_table[1]`). Los valores son `[0, 0.1]`. El valor máximo ahora es **0.1**.

Sustituyamos:
`Nuevo Q(S0, Derecha) = 0 + 0.1 * [0 + 0.9 * 0.1 - 0]`
`Nuevo Q(S0, Derecha) = 0.1 * [0.09]`
`Nuevo Q(S0, Derecha) = 0.009`

¡El valor de estar en `S0` y moverse a la derecha ha aumentado! El Agent está aprendiendo que esta Action lleva a un State (S1) que tiene un valor futuro conocido y positivo. El Reward por alcanzar el Goal se propaga lentamente hacia atrás a los States que llevan a él.

In [ ]:
q_table[0, 1] = 0.009
print("Q-Table después del Paso 3:")
print(q_table)

## Cómo se Ve en el Código

Aquí está el código Python del notebook `frozen_lake.ipynb` que implementa exactamente esta fórmula. Puedes ver cómo cada variable se corresponde directamente con la ecuación.

In [ ]:
# Del bucle de entrenamiento:
# Qtable[state][action] = Qtable[state][action] + learning_rate * (
#     reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action]
# )

# Mapeemos esto a nuestro último cálculo:
s = 0
a = 1
s_prime = 1
r = 0

learning_rate = 0.1
gamma = 0.9

# q_table antes de la actualización
q_table_before = np.array([[0., 0.], [0., 0.1], [0., 0.]])

# La actualización de Bellman en código
old_q_value = q_table_before[s, a]                 # Q(s, a)
max_future_q = np.max(q_table_before[s_prime])   # max Q(s', a')
td_error = r + gamma * max_future_q - old_q_value

new_q_value = old_q_value + learning_rate * td_error

print(f"Q-value antiguo: {old_q_value}")
print(f"Q-value futuro máximo desde S{s_prime}: {max_future_q}")
print(f"TD Error: {td_error:.3f}")
print(f"Nuevo Q-value para Q(S{s}, Derecha): {new_q_value:.3f}")

## Conclusión

Al aplicar repetidamente la Bellman Equation durante miles de episodios, el Agent explora el Environment y construye lentamente una Q-table que predice con precisión el Reward a largo plazo de cada Action desde cada State. Los pequeños valores iniciales de Reward se propagan gradualmente hacia atrás desde el State Goal, trazando un camino óptimo.

La Q-table final en `frozen_lake.ipynb` es simplemente el resultado de que este proceso ocurra miles de veces, creando una "hoja de trucos" completa para navegar el terreno helado.